<a href="https://colab.research.google.com/github/AmrBr/Reverse-Dictionary/blob/main/Reverse_Dictionary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **INSTALLS**

In [ ]:
!pip install camel-tools

### **CONSTANTS**

In [1]:
SEED = 42
SPLITS = ["train", "validation", "test"]
COLUMNS_TO_REMOVE = ['id', 'pos', 'electra', 'bertseg', 'bertmsa']
WORD = 'word'
GLOSS = 'gloss'
DS1_PATH = 'Abreekaa/Arabic-Reverse-Dictionary'
DS2_PATH = 'riotu-lab/arabic_reverse_dictionary'
CLEAN_DATA_HF_PATH = 'Abreekaa/arabic-reverse-dictionary-merged'

## **Data Preprocessing**



In [ ]:
from datasets import load_dataset

def load_data():
  dataset_1 = load_dataset(DS1_PATH)
  dataset_bulk = load_dataset(DS2_PATH)

  dataset_bulk = dataset_bulk['train']
  return dataset_1, dataset_bulk

In [ ]:
def remove_dublicates(ds1, ds2):
  seen = set()

  for split in SPLITS:
      for ex in ds1[split]:
          seen.add((ex[WORD], ex[GLOSS]))

  ds_bulk_clean = ds2.filter(lambda ex: not_duplicate(ex, seen))
  return ds_bulk_clean

def not_duplicate(ex, seen):
    return (ex[WORD], ex[GLOSS]) not in seen

def deduplicate(ds):
    seen = set()

    def is_unique(ex):
        key = (ex[WORD], ex[GLOSS])
        if key in seen:
            return False
        seen.add(key)
        return True

    return ds.filter(is_unique)

In [ ]:
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import normalize_alef_maksura_ar, normalize_alef_ar, normalize_teh_marbuta_ar

def dediacritize(ex):
  return dediac_ar(ex)

def remove_tatweel(ex):
  return ex.replace("ـ", "")

def normalize_text(ex):
  ex = normalize_alef_maksura_ar(ex)
  ex = normalize_alef_ar(ex)
  ex = normalize_teh_marbuta_ar(ex)
  return ex

In [ ]:
def clean_text(text):
    if text is None:
        return ""

    text = text.strip()
    if not text:
        return ""

    text = remove_tatweel(text)
    text = normalize_text(text)
    text = dediacritize(text)
    return text

In [ ]:
def preprocess_text(ex):
    return {
        WORD: clean_text(ex[WORD]),
        GLOSS: clean_text(ex[GLOSS]),
    }

In [ ]:
def preprocess_dataset(ds):
    ds = ds.map(preprocess_text)
    empty_count = sum(1 for ex in ds if (ex[WORD] == "" and ex[GLOSS] == ""))
    print("Empty examples: {}".format(empty_count))
    ds = ds.filter(
        lambda ex: ex[WORD] != "" and ex[GLOSS] != ""
    )
    return ds

In [ ]:
from datasets import concatenate_datasets

def merge_datasets(ds):
  return concatenate_datasets(ds)

In [ ]:
def analyze_dataset_integrity(ds, WORD_KEY='word', GLOSS_KEY='gloss'):
    splits = ['train', 'validation', 'test']

    print(f"{'Metric':<25} | {'Train':<10} | {'Val':<10} | {'Test':<10}")
    print("-" * 65)

    words = {s: set(ds[s][WORD_KEY]) for s in splits}
    pairs = {s: set(zip(ds[s][WORD_KEY], ds[s][GLOSS_KEY])) for s in splits}

    print(f"{'Total Samples':<25} | {len(ds['train']):<10} | {len(ds['validation']):<10} | {len(ds['test']):<10}")
    print(f"{'Unique Words':<25} | {len(words['train']):<10} | {len(words['validation']):<10} | {len(words['test']):<10}")
    print(f"{'Unique Pairs (W+G)':<25} | {len(pairs['train']):<10} | {len(pairs['validation']):<10} | {len(pairs['test']):<10}")

    print("\n--- Overlap Analysis (Leakage) ---")

    def print_overlap(label, set_dict):
        tr_val = len(set_dict['train'] & set_dict['validation'])
        tr_test = len(set_dict['train'] & set_dict['test'])
        print(f"{label:<25} | Train ∩ Val: {tr_val:<5} | Train ∩ Test: {tr_test}")

    print_overlap("Word Overlap", words)
    print_overlap("Full Pair Overlap", pairs)

    print("\n--- Data Integrity Insights ---")

    for s in splits:
        dupes = len(ds[s]) - len(pairs[s])
        if dupes > 0:
            print(f"{s.capitalize()}: Found {dupes} exact duplicate (Word+Gloss) rows.")

        if len(pairs[s]) > len(words[s]):
            ambiguity_count = len(pairs[s]) - len(words[s])
            print(f"{s.capitalize()}: Has {ambiguity_count} cases where one word has multiple glosses.")

    if len(pairs['test']) > 0:
        leak_pct = (len(pairs['train'] & pairs['test']) / len(pairs['test'])) * 100
        leak_pct2 = (len(pairs['validation'] & pairs['test']) / len(pairs['test'])) * 100
        print(f"\nCritical Leakage: {leak_pct:.2f}% of Test pairs are present in Training.")

In [ ]:
def build_reverse_dictionary_dataset():
  ds1, ds2 = load_data()

  # Normalize ds1
  for split in SPLITS:
      ds1[split] = ds1[split].remove_columns(COLUMNS_TO_REMOVE)
      ds1[split] = preprocess_dataset(ds1[split])
      ds1[split] = deduplicate(ds1[split])

  # Normalize ds2
  ds2 = ds2.rename_column("definition", "gloss")
  ds2 = preprocess_dataset(ds2)
  ds2 = deduplicate(ds2)

  # Remove ds2 entries already in ds1
  ds2 = remove_dublicates(ds1, ds2)

  # Merge
  dataset_full = merge_datasets([ds1["train"], ds1["test"], ds1["validation"], ds2])
  dataset_full = deduplicate(dataset_full)

  dataset_full = dataset_full.shuffle(SEED)

  train_test = dataset_full.train_test_split(test_size=0.2, seed=42)
  test_val = train_test["test"].train_test_split(test_size=0.5, seed=42)

  validation_ds = ds1["validation"]
  test_ds = ds1["test"]

  return {
      "train": train_test["train"],
      "validation": test_val["train"],
      "test": test_val["test"],
    }


In [ ]:
from datasets import DatasetDict
def save_dataset(dataset, path):
  final_ds = DatasetDict(dataset)

  final_ds.push_to_hub(path, private=True)

In [ ]:
dataset = build_reverse_dictionary_dataset()

Empty examples: 0
Empty examples: 0
Empty examples: 0
Empty examples: 0


In [ ]:
analyze_dataset_integrity(dataset)

Metric                    | Train      | Val        | Test      
-----------------------------------------------------------------
Total Samples             | 76265      | 9533       | 9534      
Unique Words              | 35310      | 7201       | 7205      
Unique Pairs (W+G)        | 76265      | 9533       | 9534      

--- Overlap Analysis (Leakage) ---
Word Overlap              | Train ∩ Val: 4060  | Train ∩ Test: 4072
Full Pair Overlap         | Train ∩ Val: 0     | Train ∩ Test: 0

--- Data Integrity Insights ---
Train: Has 40955 cases where one word has multiple glosses.
Validation: Has 2332 cases where one word has multiple glosses.
Test: Has 2329 cases where one word has multiple glosses.

Critical Leakage: 0.00% of Test pairs are present in Training.


In [ ]:
dataset

{'train': Dataset({
     features: ['word', 'gloss'],
     num_rows: 76265
 }),
 'validation': Dataset({
     features: ['word', 'gloss'],
     num_rows: 9533
 }),
 'test': Dataset({
     features: ['word', 'gloss'],
     num_rows: 9534
 })}

In [ ]:
save_dataset(dataset, CLEAN_DATA_HF_PATH)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/77 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 5.73MB / 5.73MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  722kB /  722kB            

README.md:   0%|          | 0.00/569 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


## **Model Experiments**

In [2]:
from datasets import load_dataset, concatenate_datasets
import gc

dataset = load_dataset(CLEAN_DATA_HF_PATH)
train_ds = dataset['train']
val_ds = dataset['validation']
test_ds = dataset['test']

del dataset
gc.collect()

print(f"Train examples: {len(train_ds)}, Validation: {len(val_ds)}, Test: {len(test_ds)}")

README.md:   0%|          | 0.00/569 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.73M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/723k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/722k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/76265 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9533 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9534 [00:00<?, ? examples/s]

Train examples: 76265, Validation: 9533, Test: 9534


In [6]:
from collections import defaultdict

words_length = defaultdict(int)
gloss_length = defaultdict(int)
for ex in train_ds:
  words_length[len(ex[WORD].split())] += 1
  gloss_length[len(ex[GLOSS].split())] += 1
print(words_length)
print(gloss_length)

defaultdict(<class 'int'>, {1: 56142, 2: 15476, 3: 3638, 4: 807, 6: 36, 5: 153, 8: 3, 7: 8, 13: 1, 9: 1})
defaultdict(<class 'int'>, {11: 4082, 10: 4017, 20: 1596, 15: 3442, 4: 5023, 19: 2034, 5: 5063, 13: 3763, 2: 3815, 3: 4798, 12: 4089, 6: 4325, 22: 1014, 17: 2622, 9: 4122, 14: 3691, 7: 4563, 16: 3065, 21: 1289, 18: 2456, 8: 4115, 24: 606, 25: 462, 28: 175, 35: 29, 23: 787, 27: 242, 31: 81, 29: 146, 34: 28, 32: 67, 26: 340, 30: 112, 37: 13, 39: 14, 38: 13, 80: 2, 33: 43, 36: 29, 52: 4, 43: 9, 65: 2, 62: 1, 41: 9, 68: 1, 42: 7, 53: 3, 46: 4, 115: 1, 106: 1, 49: 4, 67: 2, 47: 1, 51: 1, 50: 2, 64: 1, 48: 6, 40: 10, 54: 1, 44: 7, 89: 1, 60: 2, 56: 1, 58: 2, 55: 1, 45: 3, 57: 1, 76: 1, 77: 1, 61: 1, 69: 1})


### **TF-IDF**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.95,
    max_features=30000,
    sublinear_tf=True,
    dtype=np.float32
)

X_train = vectorizer.fit_transform(train_ds["gloss"])
train_words = list(train_ds["word"])

X_test = vectorizer.transform(test_ds["gloss"])
test_words = list(test_ds["word"])

In [ ]:
def retrieve_top_k(test_vectors, train_vectors, train_words, k=5):
    similarities = cosine_similarity(test_vectors, train_vectors)
    top_k_indices = np.argsort(-similarities, axis=1)[:, :k]
    return [[train_words[i] for i in idxs] for idxs in top_k_indices]

def evaluate(predictions, gold_words):
    top1 = sum(gold_words[i] == preds[0] for i, preds in enumerate(predictions)) / len(gold_words)
    top5 = sum(gold_words[i] in preds[:5] for i, preds in enumerate(predictions)) / len(gold_words)

    mrr = 0
    for i, preds in enumerate(predictions):
        if gold_words[i] in preds:
            mrr += 1 / (preds.index(gold_words[i]) + 1)
    mrr /= len(gold_words)

    return {"Top-1": top1, "Top-5": top5, "MRR": mrr}

In [ ]:
preds = retrieve_top_k(X_test, X_train, train_words, k=5)
metrics = evaluate(preds, test_words)
print(metrics)

### **FAISS**

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.5 MB/s eta 0:00:00


In [ ]:
import faiss

def build_faiss_index(vectors, save_path=None):
    """
    Creates a FAISS index for Cosine Similarity.
    """
    dims = vectors.shape[1]

    faiss.normalize_L2(vectors)

    index = faiss.IndexFlatIP(dims)

    index.add(vectors)

    if save_path:
        faiss.write_index(index, save_path)

    return index

### **Static Embeddings (FastText + FAISS)**

In [ ]:
!pip install fasttext

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz
!gunzip cc.ar.300.bin.gz

In [ ]:
def get_top_k_matches(query_text, model, index, k=5):

    query_vec = model.get_sentence_vector(query_text).reshape(1, -1).astype('float32')
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, k)
    return scores[0], indices[0]

In [ ]:
def vectorize_text_list(text_list, model):

    vectors = [model.get_sentence_vector(str(text)) for text in text_list]
    return np.array(vectors).astype('float32')

In [ ]:
import fasttext
import numpy as np

model = fasttext.load_model('cc.ar.300.bin')

train_vectors = vectorize_text_list(train_ds["gloss"], model)
test_vectors = vectorize_text_list(test_ds["gloss"], model)

index = build_faiss_index(train_vectors)

print(f"Starting evaluation on {len(test_ds['word'])} samples...")

top1_count = 0
top5_count = 0
mrr_sum = 0

for i, query in enumerate(test_ds["gloss"],):
    scores, indices = get_top_k_matches(query, model, index, k=5)

    predicted_words = [train_ds[int(idx)]['word'] for idx in indices]
    correct_word = test_ds['word'][i]

    if correct_word == predicted_words[0]:
        top1_count += 1

    if correct_word in predicted_words:
        top5_count += 1
        rank = predicted_words.index(correct_word) + 1
        mrr_sum += (1.0 / rank)

print(f"Top-1 Accuracy: {top1_count / len(test_ds["gloss"],):.4f}")
print(f"Top-5 Accuracy: {top5_count / len(test_ds["gloss"],):.4f}")
print(f"MRR: {mrr_sum / len(test_ds["gloss"],):.4f}")

Starting evaluation on 9534 samples...
Top-1 Accuracy: 0.1504
Top-5 Accuracy: 0.2312
MRR: 0.1817


### **Dynamic Embeddings**

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm import tqdm
import json

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

MODELS = {
    'BERTSeg': 'asafaya/bert-base-arabic',
    'AraElectra': 'aubmindlab/araelectra-base',
    'AraBERT': 'aubmindlab/bert-base-arabertv2',
    'CamelBERT': 'CAMeL-Lab/bert-base-arabic-camelbert-msa',
    'MARBERT': 'UBC-NLP/MARBERT',
    'Arabic-BERT': 'lanwuwei/Arabic-BERT',
}

FINE_TUNE_CONFIG = {
    'batch_size': 32,
    'learning_rate': 2e-5,
    'num_epochs': 3,
    'warmup_steps': 500,
    'max_length': 128,
    'temperature': 0.07,
    'negative_sample_size': 5,
}

Using device: cpu


In [ ]:
train_ds.set_format(type="pandas")
val_ds.set_format(type="pandas")
test_ds.set_format(type="pandas")

train_ds = train_ds[:]
val_ds = val_ds[:]
test_ds = test_ds[:]

In [ ]:
test_in_vocab = test_ds[test_ds['word'].isin(train_ds['word'])].reset_index(drop=True)

**Zero Shot**

In [ ]:
class ZeroShotEvaluator:
    """Evaluate pre-trained models without fine-tuning"""

    def __init__(self, model_name, model_id):
        self.model_name = model_name
        self.model_id = model_id
        self.device = DEVICE
        self.results = {}

        print(f"\nLoading {model_name}...")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_id)
            self.model = AutoModel.from_pretrained(model_id).to(self.device)
            self.model.eval()
            print(f"Loaded {model_name}")
        except Exception as e:
            print(f"Failed to load {model_name}: {e}")
            self.tokenizer = None
            self.model = None

    def get_embeddings(self, texts, batch_size=32, pooling='mean'):
        """Get embeddings for a list of texts"""
        embeddings = []

        if self.model is None:
            return None

        with torch.no_grad():
            for i in tqdm(range(0, len(texts), batch_size),
                         desc=f"Embedding {self.model_name}"):
                batch_texts = texts[i:i+batch_size]

                inputs = self.tokenizer(
                    batch_texts,
                    return_tensors='pt',
                    max_length=FINE_TUNE_CONFIG['max_length'],
                    truncation=True,
                    padding=True
                ).to(self.device)

                outputs = self.model(**inputs)

                # Different pooling strategies
                if pooling == 'mean':
                    # Mean pooling across all tokens
                    mask = inputs['attention_mask'].unsqueeze(-1).expand(
                        outputs.last_hidden_state.size()
                    ).float()
                    sum_embeddings = torch.sum(
                        outputs.last_hidden_state * mask, 1
                    )
                    sum_mask = torch.clamp(mask.sum(1), min=1e-9)
                    batch_embeddings = sum_embeddings / sum_mask
                elif pooling == 'cls':
                    # [CLS] token only
                    batch_embeddings = outputs.last_hidden_state[:, 0, :]
                elif pooling == 'max':
                    # Max pooling
                    batch_embeddings = torch.max(
                        outputs.last_hidden_state, 1
                    )[0]
                else:
                    batch_embeddings = outputs.last_hidden_state[:, 0, :]

                embeddings.append(batch_embeddings.cpu().numpy())

        return np.vstack(embeddings)

    def evaluate(self, train_data, test_data, test_iv, train_vocab, pooling='mean'):
        if self.model is None: return None

        # 1. Embeddings
        train_gloss_embs = self.get_embeddings(train_data['gloss'].tolist(), pooling=pooling)
        word_to_idx = {word: i for i, word in enumerate(train_data['word'].tolist())}
        unique_words = list(train_vocab)
        word_embs = np.array([train_gloss_embs[word_to_idx[w]] for w in unique_words])
        test_gloss_embs = self.get_embeddings(test_data['gloss'].tolist(), pooling=pooling)

        # 2. Similarities
        test_norm = test_gloss_embs / np.linalg.norm(test_gloss_embs, axis=1, keepdims=True)
        word_norm = word_embs / np.linalg.norm(word_embs, axis=1, keepdims=True)
        similarities = (test_norm @ word_norm.T)

        # 3. Calculate Ranks (This is the most efficient way)
        iv_indices = set(test_iv.index)
        word_to_vocab_idx = {word: i for i, word in enumerate(unique_words)}

        metrics = {
            'top_1': 0, 'top_5': 0, 'mrr_sum': 0.0,
            'iv_top_1': 0, 'iv_top_5': 0, 'iv_mrr_sum': 0.0
        }

        for i in range(len(test_data)):
            true_word = test_data.iloc[i]['word']
            # Find what the "correct" index should have been in our vocab list
            target_idx = word_to_vocab_idx[true_word]

            # Find the score of the correct word
            target_score = similarities[i, target_idx]

            # Rank = how many words have a HIGHER score than our target?
            # This is MUCH faster than sorting the whole array every time.
            rank = np.sum(similarities[i] > target_score) + 1

            is_iv = i in iv_indices
            rr = 1.0 / rank

            # Accumulate Overall
            if rank <= 1: metrics['top_1'] += 1
            if rank <= 5: metrics['top_5'] += 1
            metrics['mrr_sum'] += rr

            # Accumulate In-Vocab
            if is_iv:
                if rank <= 1: metrics['iv_top_1'] += 1
                if rank <= 5: metrics['iv_top_5'] += 1
                metrics['iv_mrr_sum'] += rr

        # 4. Final Formatting
        total_count = len(test_data)
        iv_count = len(test_iv)

        results = {
            'Overall': {
                'Top-1': (metrics['top_1'] / total_count) * 100,
                'Top-5': (metrics['top_5'] / total_count) * 100,
                'MRR': (metrics['mrr_sum'] / total_count)
            },
            'In-Vocab': {
                'Top-1': (metrics['iv_top_1'] / iv_count) * 100,
                'Top-5': (metrics['iv_top_5'] / iv_count) * 100,
                'MRR': (metrics['iv_mrr_sum'] / iv_count)
            }
        }

        self.results[pooling] = results
        return results

In [ ]:
def print_results(model_name, results, stage='Zero-Shot'):
    """Pretty-print results for a model"""

    print(f"\n{model_name} ({stage}):")
    print(f"  Top-1:  In-Vocab={results['top_1']['in_vocab']:6.2f}%  "
          f"Overall={results['top_1']['overall']:6.2f}%")
    print(f"  Top-5:  In-Vocab={results['top_5']['in_vocab']:6.2f}%  "
          f"Overall={results['top_5']['overall']:6.2f}%")
    print(f"  Top-10: In-Vocab={results['top_10']['in_vocab']:6.2f}%  "
          f"Overall={results['top_10']['overall']:6.2f}%")

In [ ]:
for model_name, model_id in MODELS.items():
  evaluator = ZeroShotEvaluator(model_name, model_id)
  results = evaluator.evaluate(
      train_data, test_data, test_iv, train_vocab
  )

  if results:
      print_results(model_name, results, stage='Zero-Shot')

**Fine Tuning**

### **LLMs**